In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

df = pd.read_csv('../data/processed/clean_jobs.csv')

# Add log salary as target - reduces variance effect
df['log_salary'] = np.log1p(df['salary_in_usd'])
print("Loaded:", df.shape)

Loaded: (21004, 11)


In [23]:
FEATURES = ['title_enc', 'country_enc', 'level_enc', 'category_enc', 'work_year']
TARGET = 'log_salary'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (16803, 5) Test: (4201, 5)


In [24]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train)
print("Training done!")

Training done!


In [25]:
y_pred = model.predict(X_test)

# Convert back from log scale
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)

print(f"MAE:  ${mae:,.0f}")
print(f"R2:   {r2:.4f}")

MAE:  $37,224
R2:   0.3049


In [26]:
import pickle

# Save model with log flag
with open('../model/salary_model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'log_scale': True}, f)
print("Model saved!")

Model saved!
